
# V43 LIVE — Identity-Attack Specialist

**Protected public-LB champion:** V38 = **0.608**  
**Target:** **0.620**

V42 sarcasm specialists failed the untouched validation gate, so V43 does **not** change sarcasm.

V43 targets `identity_attack`, where the LIVE validation baseline is already decent but still has enough training positives for a real specialist.

## Why this architecture?

Recent Bangla hate-speech systems have performed well with:
- Bangla-specific abusive-language encoders,
- MuRIL/Indic-family encoders,
- multitask learning,
- transformer ensembling,
- hard-negative emphasis.

V43 therefore trains two dedicated multitask experts:

1. `saroarj/BanglaHateBert`
2. `Hate-speech-CNERG/bengali-abusive-MuRIL`

Each model predicts:
- identity attack (main task)
- toxicity (auxiliary)
- profanity (auxiliary)
- threatening (auxiliary)
- sexual content (auxiliary)

Training emphasizes **toxic-but-not-identity** examples as hard negatives, because the real problem is not merely separating toxic from non-toxic text.

## Safety / generalization controls

- exact accepted LIVE split: 3297 / 500 / 1000
- grouped 5-fold CV on the 3297 training rows only
- external 500-row validation is untouched during model fitting
- no validation-based threshold optimization
- identity positive count is preserved from V38 for the safest candidates
- Google Drive fold checkpointing
- all non-identity labels remain exactly V38

At the end, upload `v43_results.zip` to ChatGPT before any Kaggle submission.


In [ ]:

# 1. Upload the V43 LIVE bundle
from google.colab import files
import pathlib, zipfile

uploaded = files.upload()
assert "v43_LIVE_identity_specialist_data.zip" in uploaded

DATA_DIR = pathlib.Path("/content/v43_data")
DATA_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile("v43_LIVE_identity_specialist_data.zip") as z:
    z.extractall(DATA_DIR)

for p in sorted(DATA_DIR.iterdir()):
    print(p.name, f"{p.stat().st_size:,} bytes")


In [ ]:

# 2. Install dependencies + preflight checkpoints
import sys, subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.6",
    "tokenizers==0.22.1",
    "sentencepiece==0.2.1",
    "scikit-learn>=1.4",
    "safetensors",
    "tqdm",
])

from transformers import AutoConfig, AutoTokenizer, AutoModel

MODELS = {
    "hatebert": "saroarj/BanglaHateBert",
    "muril": "Hate-speech-CNERG/bengali-abusive-MuRIL",
}

for key, model_name in MODELS.items():
    print("Checking:", key, model_name)
    _ = AutoConfig.from_pretrained(model_name)
    _ = AutoTokenizer.from_pretrained(model_name)

print("✅ both checkpoints accessible")


In [ ]:

# 3. Imports, config, checkpoint directory
import gc, re, math, json, random, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import average_precision_score
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

SEED = 2026
N_FOLDS = 5
EPOCHS = 3
SNAPSHOT_EPOCHS = [2,3]
MAX_LENGTH = 128

TRAIN_BATCH = 6
EVAL_BATCH = 32
ACCUM = 3

LR = {
    "hatebert": 1.35e-5,
    "muril": 1.25e-5,
}
HEAD_MULT = 3.0
LAYER_DECAY = 0.90
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

FGM_EPS = 0.50
FGM_WEIGHT = 0.30
RANK_WEIGHT = 0.15

MODELS = {
    "hatebert": "saroarj/BanglaHateBert",
    "muril": "Hate-speech-CNERG/bengali-abusive-MuRIL",
}

TASKS = [
    "identity_attack",
    "is_toxic",
    "contains_profanity",
    "threatening",
    "sexual_content",
]
MAIN = "identity_attack"

ALL_LABELS = [
    "is_toxic",
    "implicit_toxicity",
    "sarcasm",
    "contains_profanity",
    "threatening",
    "identity_attack",
    "sexual_content",
]

USE_DRIVE_CHECKPOINTS = True

if USE_DRIVE_CHECKPOINTS:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_DIR = Path("/content/drive/MyDrive/V43_Identity_Checkpoints")
else:
    RUN_DIR = Path("/content/v43_output")

RUN_DIR.mkdir(parents=True, exist_ok=True)

EXPORT_DIR = Path("/content/v43_export")
EXPORT_DIR.mkdir(exist_ok=True)

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_all(SEED)
assert torch.cuda.is_available(), "Enable a T4 GPU."

print("GPU:", torch.cuda.get_device_name(0))
print("Checkpoint directory:", RUN_DIR)


In [ ]:

# 4. Load and verify exact LIVE partition

train = pd.read_csv(DATA_DIR/"train.csv", dtype={"ID":str})
valid = pd.read_csv(DATA_DIR/"validation.csv", dtype={"ID":str})
test = pd.read_csv(DATA_DIR/"test.csv", dtype={"ID":str})
v2v = pd.read_csv(DATA_DIR/"v2_valid.csv", dtype={"ID":str})
v2t = pd.read_csv(DATA_DIR/"v2_test.csv", dtype={"ID":str})
champ = pd.read_csv(DATA_DIR/"submission_v38_champion.csv", dtype={"ID":str})
oldb = np.load(DATA_DIR/"oldb.npz")

assert len(train) == 3297
assert len(valid) == 500
assert len(test) == 1000
assert test["ID"].tolist() == champ["ID"].tolist()
assert valid["ID"].tolist() == v2v["ID"].tolist()
assert test["ID"].tolist() == v2t["ID"].tolist()
assert test["ID"].head(4).tolist() == [
    "BICH_0003","BICH_0006","BICH_0015","BICH_0016"
]

def clean_text(x):
    x = unicodedata.normalize("NFKC", str(x))
    x = x.replace("\u200c"," ").replace("\u200d"," ")
    x = re.sub(r"https?://\S+|www\.\S+", " URL ", x)
    x = re.sub(r"@\w+", " USER ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

for df in [train,valid,test]:
    df["clean_text"] = df["text"].map(clean_text)
    df["group_text"] = df["clean_text"].str.lower()

print("✅ LIVE IDs verified")
print()
print("Identity prevalence:")
print("train:", int(train[MAIN].sum()), "/", len(train), "=", round(train[MAIN].mean(),4))
print("valid:", int(valid[MAIN].sum()), "/", len(valid), "=", round(valid[MAIN].mean(),4))
print("V38 test predicted identity positives:", int(champ[MAIN].sum()))

print()
print("Identity-positive co-occurrence in TRAIN:")
display(train.loc[train[MAIN]==1, TASKS].mean().to_frame("rate"))


In [ ]:

# 5. Grouped 5-fold CV on TRAIN only

# Slightly richer stratification than identity alone:
# identity status plus toxicity status.
strata = (
    train[MAIN].astype(str)
    + "_"
    + train["is_toxic"].astype(str)
)

sgkf = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

train["fold"] = -1

for fold, (_, va_idx) in enumerate(
    sgkf.split(
        train,
        y=strata,
        groups=train["group_text"]
    )
):
    train.loc[va_idx,"fold"] = fold

assert (train["fold"] >= 0).all()
assert train.groupby("group_text")["fold"].nunique().max() == 1

display(
    train.groupby("fold").agg(
        rows=("ID","size"),
        identity_pos=(MAIN,"sum"),
        toxic_pos=("is_toxic","sum"),
    )
)


In [ ]:

# 6. Dataset, multitask model, FGM, optimizer

class MultiTaskDataset(Dataset):
    def __init__(self, df, tokenizer, labels=True):
        self.text = df["clean_text"].astype(str).tolist()
        self.y = df[TASKS].to_numpy(np.float32) if labels else None
        self.tok = tokenizer

    def __len__(self):
        return len(self.text)

    def __getitem__(self, i):
        item = self.tok(
            self.text[i],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )

        if self.y is not None:
            item["labels"] = self.y[i]

        return item


class PadCollator:
    def __init__(self, tokenizer):
        self.tok = tokenizer

    def __call__(self, items):
        labels = None

        if "labels" in items[0]:
            labels = torch.tensor(
                np.stack([x.pop("labels") for x in items]),
                dtype=torch.float32
            )

        batch = self.tok.pad(
            items,
            padding=True,
            pad_to_multiple_of=8,
            return_tensors="pt"
        )

        if labels is not None:
            batch["labels"] = labels

        return batch


class IdentityMultiTaskModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)

        if hasattr(self.encoder, "gradient_checkpointing_enable"):
            self.encoder.gradient_checkpointing_enable()

        h = self.encoder.config.hidden_size

        self.norm = nn.LayerNorm(h*2)
        self.dropout = nn.Dropout(0.20)
        self.hidden = nn.Linear(h*2, h)
        self.act = nn.GELU()
        self.classifier = nn.Linear(h, len(TASKS))

    def forward(self, **inputs):
        out = self.encoder(**inputs)
        hs = out.last_hidden_state

        mask = inputs["attention_mask"].unsqueeze(-1).float()
        mean_pool = (hs*mask).sum(1) / mask.sum(1).clamp_min(1.0)
        cls = hs[:,0]

        x = torch.cat([cls, mean_pool], dim=-1)
        x = self.norm(x)
        x = self.dropout(x)
        x = self.act(self.hidden(x))
        x = self.dropout(x)

        return self.classifier(x)


class FGM:
    def __init__(self, model, epsilon):
        self.model = model
        self.epsilon = epsilon
        self.backup = {}

    def attack(self):
        for name,param in self.model.named_parameters():
            if (
                param.requires_grad
                and param.grad is not None
                and "word_embeddings" in name
            ):
                norm = torch.norm(param.grad)

                if torch.isfinite(norm) and norm > 0:
                    self.backup[name] = param.data.clone()
                    param.data.add_(
                        self.epsilon * param.grad / norm
                    )

    def restore(self):
        for name,param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]

        self.backup = {}


def pairwise_rank_loss(main_logits, y_main):
    pos = main_logits[y_main > .5]
    neg = main_logits[y_main <= .5]

    if len(pos)==0 or len(neg)==0:
        return main_logits.new_tensor(0.0)

    if len(pos) > 16:
        pos = pos[:16]

    if len(neg) > 32:
        neg = neg[:32]

    return torch.nn.functional.softplus(
        -(pos[:,None] - neg[None,:])
    ).mean()


TASK_WEIGHTS = torch.tensor(
    [0.66, 0.14, 0.07, 0.07, 0.06],
    dtype=torch.float32
)

POS_BOOST = torch.tensor(
    [1.25, 1.00, 1.10, 1.35, 1.25],
    dtype=torch.float32
)


def multitask_loss(logits, y):
    raw = torch.nn.functional.binary_cross_entropy_with_logits(
        logits.float(),
        y,
        reduction="none"
    )

    tw = TASK_WEIGHTS.to(logits.device).unsqueeze(0)
    pb = POS_BOOST.to(logits.device).unsqueeze(0)

    weights = torch.where(
        y > .5,
        pb,
        torch.ones_like(y)
    )

    base = (raw * weights * tw).sum(1).mean()

    rank = pairwise_rank_loss(
        logits[:,0].float(),
        y[:,0]
    )

    return base + RANK_WEIGHT*rank


def build_optimizer(model, base_lr):
    named = list(model.named_parameters())

    # Works for BERT-style encoder.layer.N naming.
    rx = re.compile(r"encoder\.layer\.(\d+)")
    layer_ids = []

    for name,_ in named:
        m = rx.search(name)
        if m:
            layer_ids.append(int(m.group(1)))

    max_layer = max(layer_ids) if layer_ids else 11
    groups = {}

    for name,p in named:
        if not p.requires_grad:
            continue

        is_head = (
            name.startswith("norm")
            or name.startswith("hidden")
            or name.startswith("classifier")
        )

        m = rx.search(name)

        if is_head:
            lr = base_lr * HEAD_MULT
        elif m:
            layer = int(m.group(1))
            lr = base_lr * (LAYER_DECAY ** (max_layer-layer))
        else:
            lr = base_lr * (LAYER_DECAY ** (max_layer+1))

        no_decay = (
            name.endswith("bias")
            or "LayerNorm.weight" in name
            or "layer_norm.weight" in name
        )

        wd = 0.0 if no_decay else WEIGHT_DECAY
        key = (round(lr,12),wd)
        groups.setdefault(key,[]).append(p)

    return torch.optim.AdamW([
        {"params":ps,"lr":lr,"weight_decay":wd}
        for (lr,wd),ps in groups.items()
    ])


@torch.no_grad()
def predict(model, loader):
    model.eval()
    parts = []

    for batch in loader:
        batch.pop("labels",None)

        inputs = {
            k:v.to("cuda",non_blocking=True)
            for k,v in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits = model(**inputs)

        parts.append(
            torch.sigmoid(
                logits.float()
            ).cpu().numpy()
        )

    return np.concatenate(parts)


In [ ]:

# 7. Train one specialist with fold-level resume support

fold_seeds = [2026,17,3407,1337,42]


def train_specialist(key):
    model_name = MODELS[key]
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    collator = PadCollator(tokenizer)

    oof = np.zeros((len(train),len(TASKS)),np.float32)
    valid_parts = []
    test_parts = []
    history = []

    for fold in range(N_FOLDS):
        tr = train[train["fold"] != fold].copy()
        va = train[train["fold"] == fold].copy()

        cache = RUN_DIR/f"{key}_fold_{fold}.npz"
        hp = RUN_DIR/f"{key}_fold_{fold}_history.csv"

        if cache.exists():
            try:
                with np.load(cache) as z:
                    idx = z["idx"]
                    poof = z["oof"]
                    pvalid = z["external_valid"]
                    ptest = z["test"]

                if np.array_equal(idx,va.index.to_numpy()):
                    print("✅ reuse",key,"fold",fold)
                    oof[idx] = poof
                    valid_parts.append(pvalid)
                    test_parts.append(ptest)

                    if hp.exists():
                        history.extend(
                            pd.read_csv(hp).to_dict("records")
                        )

                    continue
            except Exception as e:
                print("Ignoring bad cache:",e)

        seed_all(fold_seeds[fold])

        # Hard-negative sampling:
        # identity positives       -> highest
        # toxic identity negatives -> strong emphasis
        # non-toxic negatives      -> downweighted
        identity = tr[MAIN].to_numpy(int)
        toxic = tr["is_toxic"].to_numpy(int)

        sample_w = np.where(
            identity==1,
            2.6,
            np.where(toxic==1,1.8,0.55)
        ).astype(float)

        sampler = WeightedRandomSampler(
            torch.tensor(sample_w,dtype=torch.double),
            num_samples=len(tr),
            replacement=True
        )

        tr_loader = DataLoader(
            MultiTaskDataset(tr,tokenizer,True),
            batch_size=TRAIN_BATCH,
            sampler=sampler,
            num_workers=0,
            pin_memory=True,
            collate_fn=collator
        )

        va_loader = DataLoader(
            MultiTaskDataset(va,tokenizer,True),
            batch_size=EVAL_BATCH,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
            collate_fn=collator
        )

        ext_loader = DataLoader(
            MultiTaskDataset(valid,tokenizer,True),
            batch_size=EVAL_BATCH,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
            collate_fn=collator
        )

        test_loader = DataLoader(
            MultiTaskDataset(test,tokenizer,False),
            batch_size=EVAL_BATCH,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
            collate_fn=collator
        )

        gc.collect()
        torch.cuda.empty_cache()

        model = IdentityMultiTaskModel(
            model_name
        ).to("cuda")

        optimizer = build_optimizer(
            model,
            LR[key]
        )

        total_steps = math.ceil(
            len(tr_loader)/ACCUM
        )*EPOCHS

        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            int(total_steps*WARMUP_RATIO),
            total_steps
        )

        scaler = torch.amp.GradScaler("cuda")
        fgm = FGM(model,FGM_EPS)

        oof_snaps = []
        ext_snaps = []
        test_snaps = []
        fold_hist = []

        for epoch in range(1,EPOCHS+1):
            model.train()
            optimizer.zero_grad(set_to_none=True)

            running = 0.0
            seen = 0

            for step,batch in enumerate(
                tqdm(
                    tr_loader,
                    desc=f"{key} f{fold} e{epoch}/{EPOCHS}"
                )
            ):
                y = batch.pop("labels").to(
                    "cuda",
                    non_blocking=True
                )

                inputs = {
                    k:v.to("cuda",non_blocking=True)
                    for k,v in batch.items()
                }

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16
                ):
                    logits = model(**inputs)

                loss = multitask_loss(
                    logits,
                    y
                )

                if not torch.isfinite(loss):
                    raise FloatingPointError(
                        f"non-finite loss: {key} fold {fold}"
                    )

                scaler.scale(
                    loss/ACCUM
                ).backward()

                if epoch >= 2:
                    fgm.attack()

                    with torch.autocast(
                        device_type="cuda",
                        dtype=torch.float16
                    ):
                        adv_logits = model(**inputs)

                    adv_loss = multitask_loss(
                        adv_logits,
                        y
                    )

                    scaler.scale(
                        FGM_WEIGHT*adv_loss/ACCUM
                    ).backward()

                    fgm.restore()

                running += float(loss.item())*len(y)
                seen += len(y)

                if (
                    (step+1)%ACCUM==0
                    or step+1==len(tr_loader)
                ):
                    scaler.unscale_(optimizer)

                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(),
                        1.0
                    )

                    old_scale = scaler.get_scale()
                    scaler.step(optimizer)
                    scaler.update()

                    if scaler.get_scale() >= old_scale:
                        scheduler.step()

                    optimizer.zero_grad(set_to_none=True)

            rec = {
                "model":key,
                "fold":fold,
                "epoch":epoch,
                "loss":running/max(seen,1),
            }

            fold_hist.append(rec)
            history.append(rec)

            pd.DataFrame(fold_hist).to_csv(
                hp,index=False
            )

            print(rec)

            if epoch in SNAPSHOT_EPOCHS:
                oof_snaps.append(
                    predict(model,va_loader)
                )
                ext_snaps.append(
                    predict(model,ext_loader)
                )
                test_snaps.append(
                    predict(model,test_loader)
                )

        poof = np.mean(oof_snaps,axis=0)
        pvalid = np.mean(ext_snaps,axis=0)
        ptest = np.mean(test_snaps,axis=0)

        idx = va.index.to_numpy()
        oof[idx] = poof
        valid_parts.append(pvalid)
        test_parts.append(ptest)

        np.savez_compressed(
            cache,
            idx=idx,
            oof=poof,
            external_valid=pvalid,
            test=ptest
        )

        del model,optimizer,scheduler,scaler
        del tr_loader,va_loader,ext_loader,test_loader
        gc.collect()
        torch.cuda.empty_cache()

    pvalid = np.mean(valid_parts,axis=0)
    ptest = np.mean(test_parts,axis=0)

    np.savez_compressed(
        RUN_DIR/f"{key}_complete.npz",
        oof=oof,
        external_valid=pvalid,
        test=ptest
    )

    pd.DataFrame(history).to_csv(
        RUN_DIR/f"{key}_history.csv",
        index=False
    )

    return oof,pvalid,ptest


In [ ]:

# 8. Train both identity specialists

hate_oof,hate_valid,hate_test = train_specialist(
    "hatebert"
)

muril_oof,muril_valid,muril_test = train_specialist(
    "muril"
)

print("✅ V43 model training complete")


In [ ]:

# 9. Train-OOF-only ensemble selection

def logit(p):
    p = np.clip(np.asarray(p,float),1e-6,1-1e-6)
    return np.log(p/(1-p))

def sigmoid(z):
    return 1/(1+np.exp(-z))

def topk(prob,k):
    out = np.zeros(len(prob),int)
    k = int(max(0,min(k,len(prob))))

    if k:
        out[np.argsort(-prob)[:k]] = 1

    return out

def f1_pos(y,p):
    y = np.asarray(y,int)
    p = np.asarray(p,int)

    tp = ((y==1)&(p==1)).sum()
    fp = ((y==0)&(p==1)).sum()
    fn = ((y==1)&(p==0)).sum()

    d = 2*tp+fp+fn
    return float(2*tp/d) if d else 0.0


y_train = train[MAIN].to_numpy(int)

best = None

# Main identity index = 0, toxic auxiliary index = 1.
for w_hate in [0.0,0.25,0.50,0.75,1.0]:
    identity_score = sigmoid(
        w_hate*logit(hate_oof[:,0])
        +(1-w_hate)*logit(muril_oof[:,0])
    )

    toxic_score = sigmoid(
        w_hate*logit(hate_oof[:,1])
        +(1-w_hate)*logit(muril_oof[:,1])
    )

    for toxic_gate in [0.0,0.10,0.20]:
        score = sigmoid(
            logit(identity_score)
            + toxic_gate*logit(toxic_score)
        )

        ap = average_precision_score(
            y_train,
            score
        )

        rec = {
            "ap":float(ap),
            "w_hate":float(w_hate),
            "w_muril":float(1-w_hate),
            "toxic_gate":float(toxic_gate),
        }

        if best is None or rec["ap"] > best["ap"]:
            best = rec

print("OOF-selected recipe:")
print(json.dumps(best,indent=2))


In [ ]:

# 10. Build external-validation and test specialist scores

w = best["w_hate"]
g = best["toxic_gate"]

new_valid_identity = sigmoid(
    w*logit(hate_valid[:,0])
    +(1-w)*logit(muril_valid[:,0])
)

new_test_identity = sigmoid(
    w*logit(hate_test[:,0])
    +(1-w)*logit(muril_test[:,0])
)

new_valid_toxic = sigmoid(
    w*logit(hate_valid[:,1])
    +(1-w)*logit(muril_valid[:,1])
)

new_test_toxic = sigmoid(
    w*logit(hate_test[:,1])
    +(1-w)*logit(muril_test[:,1])
)

new_valid_score = sigmoid(
    logit(new_valid_identity)
    +g*logit(new_valid_toxic)
)

new_test_score = sigmoid(
    logit(new_test_identity)
    +g*logit(new_test_toxic)
)

# Reconstruct V14/V38 identity probability.
# V38 did not change identity_attack from V14.
IDENTITY_INDEX_ALL = 5

base_valid_score = (
    .70*v2v["identity_attack_prob"].to_numpy(float)
    +.30*oldb["valid"][2,:,IDENTITY_INDEX_ALL]
)

base_test_score = (
    .70*v2t["identity_attack_prob"].to_numpy(float)
    +.30*oldb["test"][2,:,IDENTITY_INDEX_ALL]
)

BASE_THRESHOLD = 0.3814

base_valid_pred = (
    base_valid_score >= BASE_THRESHOLD
).astype(int)

base_test_pred = champ[MAIN].to_numpy(int)

assert int(base_test_pred.sum()) == int(champ[MAIN].sum())

# Candidate A: pure new specialist, fixed V38 counts
pure_valid_pred = topk(
    new_valid_score,
    int(base_valid_pred.sum())
)

pure_test_pred = topk(
    new_test_score,
    int(base_test_pred.sum())
)

# Candidate B: conservative 70% V14/V38 rank + 30% new specialist
safe_valid_score = sigmoid(
    .70*logit(base_valid_score)
    +.30*logit(new_valid_score)
)

safe_test_score = sigmoid(
    .70*logit(base_test_score)
    +.30*logit(new_test_score)
)

safe_valid_pred = topk(
    safe_valid_score,
    int(base_valid_pred.sum())
)

safe_test_pred = topk(
    safe_test_score,
    int(base_test_pred.sum())
)

# Candidate C: 50/50 rank blend, also fixed count
mid_valid_score = sigmoid(
    .50*logit(base_valid_score)
    +.50*logit(new_valid_score)
)

mid_test_score = sigmoid(
    .50*logit(base_test_score)
    +.50*logit(new_test_score)
)

mid_valid_pred = topk(
    mid_valid_score,
    int(base_valid_pred.sum())
)

mid_test_pred = topk(
    mid_test_score,
    int(base_test_pred.sum())
)


In [ ]:

# 11. Untouched 500-row validation gate
#
# This is the first cell that uses the validation identity labels.

y_valid = valid[MAIN].to_numpy(int)

baseline_f1 = f1_pos(
    y_valid,
    base_valid_pred
)

baseline_ap = average_precision_score(
    y_valid,
    base_valid_score
)

candidate_map = {
    "safe30_fixed_count": (
        safe_valid_pred,
        safe_test_pred,
        safe_valid_score,
    ),
    "mid50_fixed_count": (
        mid_valid_pred,
        mid_test_pred,
        mid_valid_score,
    ),
    "pure_specialist_fixed_count": (
        pure_valid_pred,
        pure_test_pred,
        new_valid_score,
    ),
}

rows = []

for name,(pv,pt,sv) in candidate_map.items():
    f1 = f1_pos(y_valid,pv)
    ap = average_precision_score(y_valid,sv)

    rows.append({
        "candidate":name,
        "validation_f1":f1,
        "validation_gain":f1-baseline_f1,
        "validation_ap":ap,
        "ap_gain":ap-baseline_ap,
        "validation_positive_count":int(pv.sum()),
        "test_positive_count":int(pt.sum()),
        "test_changes_vs_v38":int(
            (pt != base_test_pred).sum()
        ),
    })

report = pd.DataFrame(rows).sort_values(
    ["validation_f1","validation_ap"],
    ascending=False
).reset_index(drop=True)

print("V38 identity validation F1:",baseline_f1)
print("V38 identity validation AP:",baseline_ap)
display(report.round(6))

best_row = report.iloc[0].to_dict()

# Strong gate because V24's smaller identity gains did not transfer.
PASS = bool(
    best_row["validation_gain"] >= .030
    and best_row["ap_gain"] >= .020
    and best_row["test_changes_vs_v38"] <= 180
)

summary = {
    "protected_public_lb":0.608,
    "target_public_lb":0.620,
    "baseline_identity_f1":float(baseline_f1),
    "baseline_identity_ap":float(baseline_ap),
    "oof_recipe":best,
    "best_candidate":best_row,
    "PASS":PASS,
}

print(json.dumps(summary,indent=2))


In [ ]:

# 12. Create candidate submissions
#
# Only identity_attack may change. All other V38 labels are frozen.

submission_paths = {}

for name,(_,pt,_) in candidate_map.items():
    sub = champ.copy()
    sub[MAIN] = pt.astype(int)

    assert sub["ID"].tolist() == test["ID"].tolist()
    assert sub["ID"].head(4).tolist() == [
        "BICH_0003","BICH_0006","BICH_0015","BICH_0016"
    ]

    for label in ALL_LABELS:
        if label != MAIN:
            assert np.array_equal(
                sub[label].to_numpy(int),
                champ[label].to_numpy(int)
            )

    assert sub[ALL_LABELS].isin([0,1]).all().all()

    path = EXPORT_DIR/f"submission_v43_{name}.csv"
    sub.to_csv(path,index=False)
    submission_paths[name] = str(path)

recommended_name = report.iloc[0]["candidate"]

if PASS:
    recommended = pd.read_csv(
        submission_paths[recommended_name],
        dtype={"ID":str}
    )

    recommended_path = EXPORT_DIR/"submission_v43_RECOMMENDED.csv"
    recommended.to_csv(recommended_path,index=False)

    print("✅ V43 PASSED")
    print("Recommended:",recommended_name)
    print("Send v43_results.zip to ChatGPT before Kaggle.")
else:
    print("❌ V43 did not pass the strict gate.")
    print("Do NOT submit. Send v43_results.zip for analysis.")


In [ ]:

# 13. Package results

np.savez_compressed(
    EXPORT_DIR/"v43_identity_probabilities.npz",
    hate_oof=hate_oof,
    hate_valid=hate_valid,
    hate_test=hate_test,
    muril_oof=muril_oof,
    muril_valid=muril_valid,
    muril_test=muril_test,
    new_valid_score=new_valid_score,
    new_test_score=new_test_score,
    base_valid_score=base_valid_score,
    base_test_score=base_test_score,
)

report.to_csv(
    EXPORT_DIR/"v43_validation_report.csv",
    index=False
)

with open(EXPORT_DIR/"v43_summary.json","w") as f:
    json.dump(summary,f,indent=2)

zip_path = Path("/content/v43_results.zip")

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:
    for p in EXPORT_DIR.iterdir():
        if p.is_file():
            z.write(p,arcname=p.name)

print("Created:",zip_path)

from google.colab import files
files.download(str(zip_path))
